In [1]:
import pyspark




In [2]:
spark = pyspark.sql.SparkSession.builder.appName("csv").config("spark.driver.maxResultSize", "16g") \
    .config("spark.driver.memory", "32g").getOrCreate()
spark.conf.set("spark.sql.caseSensitive", "true")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/13 10:16:14 WARN Utils: Your hostname, rocinante, resolves to a loopback address: 127.0.1.1; using 192.168.0.226 instead (on interface enp37s0f0)
26/10/13 10:16:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/13 10:16:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# 2. Read the Parquet file or directory
# You can point this to a single .parquet file or a folder containing partitioned parquet files
df = spark.read.parquet("../silver/idnet100k_no_dupe.parquet")



In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Create a global window (Warning: moves all data to 1 partition temporarily to index it)
window_spec = Window.orderBy(F.lit(1))

# 2. Generate actual sequential row numbers (1 to 100,000)
df_sequential = df.withColumn("row_idx", F.row_number().over(window_spec) - 1)

# 3. Mathematically divide into chunks of 20,000
df_chunks = df_sequential.withColumn("chunk_id", F.floor(F.col("row_idx") / 10000))

df_chunks = df_chunks.drop("row_idx")

# 4. Write partitioned by the chunk id, and coalesce to 1 file per partition folder
df_chunks.coalesce(1).write \
    .mode("overwrite") \
    .partitionBy("chunk_id") \
    .option("header", "true") \
    .option("quoteAll", "true") \
    .csv("../silver/idnet100k_no_dupe.csv")


26/10/13 10:16:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/13 10:16:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/13 10:16:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/13 10:21:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/13 10:21:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [ ]:
# 3. Write the DataFrame to CSV
# 'header=True' ensures column names are included as the first row
df.coalesce(1).write.mode("overwrite").option("header", "true").csv("../silver/idnet100k_no_dupe.csv")

In [ ]:
df = spark.read.parquet("../silver/idnet10k_no_dupe.parquet")

In [ ]:
df.coalesce(1).write.mode("overwrite").option("header", "true").csv("../silver/idnet10k_no_dupe.csv")

In [ ]:
df = spark.read.parquet("../silver/idnet_no_dupe.parquet")

In [ ]:
df.coalesce(1).write.mode("overwrite").option("header", "true").csv("../silver/idnet_no_dupe.csv")

In [ ]:
df = spark.read.parquet("../silver/alt_docs_x5_no_dupe.parquet")

In [ ]:
df.coalesce(1).write.mode("overwrite").option("header", "true").csv("../silver/alt_docs_x5_no_dupe.csv")